## Why storage matters — the container filesystem is ephemeral

When a container exits and the kubelet restarts it, the writable filesystem is **wiped**:

- The image's read-only layers stay (cached on the node).
- The container's writable top layer is **deleted** when the container is removed.

By design — containers are cattle, not pets. But it means anything written to `/var/lib/postgres`, `/data`, or any "let me just save this here" path is gone on restart:

- **Database files** → restarts wipe your data.
- **Uploaded files** → restarts wipe your uploads.
- **Caches** → wiped (sometimes fine, sometimes not).

You need storage that *outlives the container*. Kubernetes offers two lifetimes:

- **Pod-lifetime storage** — a volume that lives as long as the *pod*: `emptyDir`, ConfigMap/Secret volumes. Shared between the pod's containers, deleted with the pod. Scratch space, sidecar IPC, mounted config.
- **Cluster-lifetime storage** — backed by something *external* to the cluster (a cloud disk, NFS, a SAN LUN) that survives pod deletion, node failure, and rescheduling. That's the **PV / PVC** abstraction.

For each workload the question is simply: **how long should this data live?** — and the answer picks the volume type. On our map, this is why the **Storage** box exists at all: the Pods box up top is ephemeral, so durable state has to live in a separate resource the Pod mounts.